# WeatherForYnov — LSTM Multi-Sorties (Temp. Moy / Max / Min)

**Hackathon YNOV — Sujet 2 : Prévisions météorologiques**

Ce notebook entraîne **3 modèles LSTM indépendants** pour prédire sur les 12 prochains mois :
- `temp_moy_quotidienne` — Moyenne mensuelle des températures moyennes journalières (TMM)
- `temp_max_quotidienne` — Moyenne mensuelle des températures maximales journalières (TX)
- `temp_min_quotidienne` — Moyenne mensuelle des températures minimales journalières (TN)

## Configuration du modèle
| Paramètre | Valeur |
|-----------|--------|
| Source données | `climat_lstm_targets.csv` (issu du notebook 03) |
| Fenêtre d'entrée | **36 mois** (3 ans d'historique) |
| Horizon de sortie | **12 mois** |
| Architecture | LSTM(128) → Dropout(0.2) → LSTM(64) → Dropout(0.2) → Dense(12) |
| Stratégie | **3 modèles indépendants** (1 par cible) |
| Train | 2001–2022 |
| Test | 2023–2025 |
| Loss | Huber |
| Périmètre | 100 stations stratifiées (train) / toutes stations (test) |
| Batch size | **512 CPU / 256 GPU** (auto-détecté) |
| Max epochs | **100** (EarlyStopping patience=20) |
| CPU optim | oneDNN/MKL · 16 threads · KMP_AFFINITY |
| GPU optim | Intel Arc auto-détecté (DirectML/ITEX) |
| Mémoire | Mixed float16 (GPU) · free_memory() entre modèles |
| Pipeline | tf.data + cache + prefetch(AUTOTUNE) |

## Métriques d'évaluation
**MAE** (°C) · **RMSE** (°C) · **MAPE** (%) · **R²** — par cible et par horizon

---
## 0. (Optionnel) Activation du GPU Intel Arc

Exécute cette cellule **une seule fois** pour installer le plugin Intel GPU pour TensorFlow.
Après installation, **redémarre le kernel** — TensorFlow utilisera l'Intel Arc automatiquement.
Si l'installation échoue, ignore-la : le notebook fonctionne parfaitement en CPU.

In [ ]:
import subprocess, sys

print("Installation du plugin Intel Extension for TensorFlow (GPU Intel Arc)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "intel-extension-for-tensorflow[gpu]", "-q"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✓ Plugin installé — redémarre le kernel puis relance depuis la cellule 1.")
    print("  TensorFlow détectera automatiquement l'Intel Arc comme device GPU.")
else:
    print("✗ Installation échouée (normal sans drivers Intel OneAPI installés).")
    print("  Le notebook tourne en CPU + mixed_float16, c'est suffisant.")

In [11]:
import os
import gc
import warnings
from pathlib import Path

# ═══════════════════════════════════════════════════════════════════
# BLOC 1 — OPTIMISATIONS CPU (Intel Core Ultra 7 165H)
# Doit être défini AVANT l'import de tensorflow
# ═══════════════════════════════════════════════════════════════════
N_PHYSICAL_CORES = 16  # Core Ultra 7 165H : 6P + 8E + 2LP cores

os.environ["TF_ENABLE_ONEDNN_OPTS"]       = "1"    # oneDNN/MKL activé
os.environ["OMP_NUM_THREADS"]             = str(N_PHYSICAL_CORES)
os.environ["TF_NUM_INTRAOP_THREADS"]      = str(N_PHYSICAL_CORES)
os.environ["TF_NUM_INTEROP_THREADS"]      = "4"
os.environ["KMP_BLOCKTIME"]               = "1"    # réduit l'attente entre threads
os.environ["KMP_SETTINGS"]               = "1"
os.environ["KMP_AFFINITY"]               = "granularity=fine,compact,1,0"
os.environ["TF_CPP_MIN_LOG_LEVEL"]        = "2"    # supprime les warnings TF verbeux

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, mixed_precision

# ═══════════════════════════════════════════════════════════════════
# BLOC 2 — OPTIMISATIONS GPU & MÉMOIRE
# ═══════════════════════════════════════════════════════════════════

# Threading TensorFlow (renforce les vars d'env)
tf.config.threading.set_intra_op_parallelism_threads(N_PHYSICAL_CORES)
tf.config.threading.set_inter_op_parallelism_threads(4)

# Détection GPU (Intel Arc via DirectML ou ITEX, CUDA, etc.)
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✓ GPU détecté et configuré : {[g.name for g in gpus]}")
        USE_GPU = True
    except RuntimeError as e:
        print(f"⚠ GPU détecté mais erreur config : {e}")
        USE_GPU = False
else:
    print("ℹ Aucun GPU TensorFlow détecté — exécution en CPU optimisé (oneDNN/MKL)")
    USE_GPU = False

# Mixed Precision float16 (divise par ~2 l'usage mémoire + accélère les matmul)
# Utiliser float32 sur CPU pur si pas de GPU (évite instabilités numériques)
if USE_GPU:
    mixed_precision.set_global_policy("mixed_float16")
    print(f"✓ Mixed precision : {mixed_precision.global_policy().name}")
else:
    mixed_precision.set_global_policy("float32")
    print(f"ℹ Precision : float32 (CPU mode)")

# ═══════════════════════════════════════════════════════════════════
# BLOC 3 — UTILITAIRE GESTION MÉMOIRE ENTRE MODÈLES
# ═══════════════════════════════════════════════════════════════════
def free_memory(label: str = ""):
    """Libère la RAM et le cache GPU entre chaque modèle entraîné."""
    gc.collect()
    tf.keras.backend.clear_session()
    if label:
        print(f"  [RAM libérée après : {label}]")

# ═══════════════════════════════════════════════════════════════════
# BLOC 4 — CONFIG GÉNÉRALE
# ═══════════════════════════════════════════════════════════════════
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"]    = 100

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SOURCE_CSV  = ROOT / "src" / "data" / "processed" / "climat_lstm_targets.csv"
MODELS_DIR  = ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyperparamètres ───────────────────────────────────────────────
WINDOW      = 36
HORIZON     = 12
TRAIN_END   = 2022
TEST_START  = 2023
BATCH_SIZE  = 512 if not USE_GPU else 256  # GPU → batch plus petit, pipeline plus efficace
MAX_EPOCHS  = 100
LSTM_UNITS  = [128, 64]
DROPOUT     = 0.2

# ── Sous-échantillonnage des stations ─────────────────────────────
MAX_TRAIN_STATIONS = 100   # 100 stations stratifiées ≈ 15k séquences ≈ ~30 steps/époque
MAX_TRAIN_SEQ      = 15_000

# ── Cibles ────────────────────────────────────────────────────────
TARGETS = [
    "temp_moy_quotidienne",
    "temp_max_quotidienne",
    "temp_min_quotidienne",
]

print(f"\nTensorFlow    : {tf.__version__}")
print(f"Source CSV    : {SOURCE_CSV}")
print(f"Fenêtre       : {WINDOW} mois  |  Horizon : {HORIZON} mois")
print(f"Train         : 2001–{TRAIN_END}  |  Test : {TEST_START}–2025")
print(f"Batch size    : {BATCH_SIZE}")
print(f"oneDNN/MKL    : {os.environ.get('TF_ENABLE_ONEDNN_OPTS', '0') == '1'}")

TensorFlow : 2.20.0
Source CSV  : c:\Users\MASTER DATA&IA\Desktop\YNOV\Hackathon\src\data\processed\climat_lstm_targets.csv
Fenêtre     : 36 mois  |  Horizon : 12 mois
Train       : 2001–2022  |  Test  : 2023–2025


---
## 2. Chargement et préparation des données

In [12]:
df_raw = pd.read_csv(SOURCE_CSV, parse_dates=["date"])
print(f"Dimensions brutes : {df_raw.shape}")
print(f"Période           : {df_raw['annee'].min()} – {df_raw['annee'].max()}")
print(f"Stations totales  : {df_raw['NUM_POSTE'].nunique():,}")

# ── Sous-échantillonnage stratifié des stations ───────────────────
# Objectif : garder MAX_TRAIN_STATIONS stations repr. de toutes les régions.
# Les stations du TEST ne sont pas affectées (on évalue sur toutes).
if MAX_TRAIN_STATIONS is not None and df_raw["NUM_POSTE"].nunique() > MAX_TRAIN_STATIONS:
    dep_col = "code_departement" if "code_departement" in df_raw.columns else None

    if dep_col:
        # Tirage stratifié par département
        stations_per_dep = (
            df_raw.drop_duplicates("NUM_POSTE")
            .groupby(dep_col)["NUM_POSTE"]
            .apply(list)
        )
        rng_sub = np.random.default_rng(SEED)
        selected = []
        total_dep = len(stations_per_dep)
        for dep, stlist in stations_per_dep.items():
            n_draw = max(1, round(MAX_TRAIN_STATIONS * len(stlist) /
                                  df_raw["NUM_POSTE"].nunique()))
            selected.extend(rng_sub.choice(stlist,
                                           size=min(n_draw, len(stlist)),
                                           replace=False).tolist())
        # Ajustement si on dépasse ou manque légèrement la cible
        all_stations = df_raw["NUM_POSTE"].unique().tolist()
        remaining = [s for s in all_stations if s not in set(selected)]
        while len(selected) < MAX_TRAIN_STATIONS and remaining:
            pick = rng_sub.choice(remaining)
            selected.append(pick)
            remaining.remove(pick)
        selected = selected[:MAX_TRAIN_STATIONS]
    else:
        # Tirage simple si pas de département
        all_stations = df_raw["NUM_POSTE"].unique()
        selected = np.random.default_rng(SEED).choice(
            all_stations, size=MAX_TRAIN_STATIONS, replace=False
        ).tolist()

    TRAIN_STATIONS = set(selected)
    print(f"\nSous-échantillonnage stratifié : {len(TRAIN_STATIONS)} stations retenues pour le TRAIN")
    print(f"(toutes les stations restent utilisées pour l'évaluation TEST)")
else:
    TRAIN_STATIONS = set(df_raw["NUM_POSTE"].unique())
    print(f"\nToutes les stations utilisées pour le TRAIN : {len(TRAIN_STATIONS):,}")

display(df_raw.head(3))

Dimensions brutes : (304575, 32)
Période           : 2001 – 2025
Stations totales  : 1,331

Sous-échantillonnage stratifié : 600 stations retenues pour le TRAIN
(toutes les stations restent utilisées pour l'évaluation TEST)


,NUM_POSTE,nom_station,lat,lon,alti,code_departement,annee,mois,date,temp_moy_quotidienne,...,nb_jours_tn10,precipitations_mm,precip_max_24h,nb_jours_pluie,humidite,vent_moyen,rafale_max,insolation_min,pression_mer,evapotranspiration
0,1014002,ARBENT,46.278167,5.669,534,1,2004,3,2004-03-01,0.54,...,0.0,2.38,0.78,5.0,78.0,0.21,1.4,NaN,NaN,NaN
1,1014002,ARBENT,46.278167,5.669,534,1,2004,5,2004-05-01,1.10,...,0.0,9.50,3.98,13.0,74.0,0.19,1.4,NaN,NaN,9.54
2,1014002,ARBENT,46.278167,5.669,534,1,2004,6,2004-06-01,1.65,...,0.0,5.63,2.24,6.0,70.0,0.21,1.8,NaN,NaN,11.41


In [13]:
# Features d'entrée candidates (hors cibles)
CANDIDATE_FEATURES = [
    "temp_max_absolu", "temp_min_absolu",
    "temp_moy_max", "temp_moy_min",
    "amplitude_thermique",
    "nb_jours_tx25", "nb_jours_tx30",
    "nb_jours_gelee", "nb_jours_tn5",
    "precipitations_mm", "nb_jours_pluie",
    "humidite", "vent_moyen",
    "insolation_min", "pression_mer", "evapotranspiration",
]

# Garder uniquement les colonnes présentes dans le CSV
FEATURES = TARGETS + [f for f in CANDIDATE_FEATURES if f in df_raw.columns]
print(f"Features retenues : {len(FEATURES)}")
print(FEATURES)

# Sélection et tri
KEEP_COLS = ["NUM_POSTE", "nom_station", "annee", "mois", "date"] + FEATURES
KEEP_COLS = [c for c in KEEP_COLS if c in df_raw.columns]

df = df_raw[KEEP_COLS].sort_values(["NUM_POSTE", "date"]).reset_index(drop=True)

Features retenues : 19
['temp_moy_quotidienne', 'temp_max_quotidienne', 'temp_min_quotidienne', 'temp_max_absolu', 'temp_min_absolu', 'temp_moy_max', 'temp_moy_min', 'amplitude_thermique', 'nb_jours_tx25', 'nb_jours_tx30', 'nb_jours_gelee', 'nb_jours_tn5', 'precipitations_mm', 'nb_jours_pluie', 'humidite', 'vent_moyen', 'insolation_min', 'pression_mer', 'evapotranspiration']


In [14]:
# Imputation par interpolation linéaire par station, puis médiane globale
df[FEATURES] = df.groupby("NUM_POSTE")[FEATURES].transform(
    lambda x: x.interpolate(method="linear", limit_direction="both")
)
for col in FEATURES:
    df[col] = df[col].fillna(df[col].median())

print(f"Valeurs manquantes restantes : {df[FEATURES].isnull().sum().sum()}")

Valeurs manquantes restantes : 0


---
## 3. Feature Engineering

In [15]:
# Encodage cyclique du mois
df["mois_sin"] = np.sin(2 * np.pi * df["mois"] / 12)
df["mois_cos"] = np.cos(2 * np.pi * df["mois"] / 12)

# Lags par station pour chaque cible (capture la dynamique saisonnière)
lag_cols = []
for target in TARGETS:
    short = target.split("_")[1]  # "moy", "max", "min"
    for lag in [1, 3, 6, 12]:
        col = f"{short}_lag_{lag}"
        df[col] = df.groupby("NUM_POSTE")[target].shift(lag)
        lag_cols.append(col)

# Rolling 12 mois sur la température moyenne (tendance annuelle)
df["tmm_roll12_mean"] = (
    df.groupby("NUM_POSTE")["temp_moy_quotidienne"]
    .transform(lambda x: x.rolling(12, min_periods=1).mean())
)
df["tmm_roll12_std"] = (
    df.groupby("NUM_POSTE")["temp_moy_quotidienne"]
    .transform(lambda x: x.rolling(12, min_periods=1).std().fillna(0))
)

# Remplissage des NaN créés par les lags
df[lag_cols] = df.groupby("NUM_POSTE")[lag_cols].transform(
    lambda x: x.fillna(method="bfill").fillna(method="ffill")
)

# Liste finale de toutes les features d'entrée du modèle
ALL_FEATURES = (
    FEATURES
    + ["mois_sin", "mois_cos"]
    + lag_cols
    + ["tmm_roll12_mean", "tmm_roll12_std"]
)

N_FEATURES = len(ALL_FEATURES)
print(f"Nombre total de features d'entrée : {N_FEATURES}")

Nombre total de features d'entrée : 35


---
## 4. Normalisation par station et création des séquences

In [16]:
scalers = {}   # scalers[poste][target] = MinMaxScaler ajusté sur le train
df_scaled_list = []

for poste, grp in df.groupby("NUM_POSTE"):
    grp = grp.sort_values("date").copy()
    train_mask = grp["annee"] <= TRAIN_END

    scalers[poste] = {}

    # Scaler des features (ajusté sur le train)
    scaler_X = MinMaxScaler(feature_range=(-1, 1))
    scaler_X.fit(grp.loc[train_mask, ALL_FEATURES])
    grp[ALL_FEATURES] = scaler_X.transform(grp[ALL_FEATURES])
    scalers[poste]["X"] = scaler_X

    # Scaler individuel pour chaque cible
    for target in TARGETS:
        sc = MinMaxScaler(feature_range=(-1, 1))
        sc.fit(grp.loc[train_mask, [target]])
        scalers[poste][target] = sc

    df_scaled_list.append(grp)

df_scaled = pd.concat(df_scaled_list, ignore_index=True)
print(f"Données normalisées : {df_scaled.shape}")
print(f"Scalers créés pour {len(scalers)} stations")

Données normalisées : (304575, 40)
Scalers créés pour 1331 stations


In [17]:
def make_sequences(group_df, window, horizon, feature_cols, target_col):
    """Séquences glissantes : X = (window × features), y = (horizon valeurs cible)."""
    X, y = [], []
    feat = group_df[feature_cols].values
    tgt  = group_df[target_col].values
    for i in range(len(feat) - window - horizon + 1):
        X.append(feat[i : i + window])
        y.append(tgt[i + window : i + window + horizon])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


# Dictionnaire : train_data[target] = (X_train, y_train)
#               test_data[target]  = (X_test, y_test, meta)
train_data = {t: ([], []) for t in TARGETS}
test_data  = {t: ([], [], []) for t in TARGETS}

for poste, grp in df_scaled.groupby("NUM_POSTE"):
    grp = grp.sort_values("date").reset_index(drop=True)
    train_grp = grp[grp["annee"] <= TRAIN_END].copy()

    for target in TARGETS:
        # Train — uniquement les stations sélectionnées (sous-échantillonnage)
        if poste in TRAIN_STATIONS and len(train_grp) >= WINDOW + HORIZON:
            X_tr, y_tr = make_sequences(train_grp, WINDOW, HORIZON, ALL_FEATURES, target)
            train_data[target][0].append(X_tr)
            train_data[target][1].append(y_tr)

        # Test : toutes les stations évaluées (pas de filtre)
        # Fenêtre couvrant les WINDOW derniers mois du train + début du test
        cutoff = TRAIN_END - (WINDOW // 12) - 1
        test_grp = grp[grp["annee"] >= cutoff].copy().reset_index(drop=True)
        if len(test_grp) >= WINDOW + HORIZON:
            X_te, y_te = make_sequences(test_grp, WINDOW, HORIZON, ALL_FEATURES, target)
            # Ne garder que les séquences qui prédisent sur la période test
            for j in range(len(X_te)):
                pred_start = WINDOW + j
                if pred_start < len(test_grp):
                    yr = test_grp.loc[pred_start, "annee"] if pred_start in test_grp.index else None
                    if yr is not None and yr >= TEST_START:
                        test_data[target][0].append(X_te[j])
                        test_data[target][1].append(y_te[j])
                        test_data[target][2].append({"NUM_POSTE": poste})
                        break

# Consolidation
X_trains, y_trains, X_tests, y_tests = {}, {}, {}, {}
test_metas = {}

rng_seq = np.random.default_rng(SEED)
for target in TARGETS:
    X_tr = np.concatenate(train_data[target][0], axis=0)
    y_tr = np.concatenate(train_data[target][1], axis=0)

    # Cap sur le nombre total de séquences (sécurité RAM)
    if len(X_tr) > MAX_TRAIN_SEQ:
        idx_cap = rng_seq.choice(len(X_tr), size=MAX_TRAIN_SEQ, replace=False)
        X_tr, y_tr = X_tr[idx_cap], y_tr[idx_cap]

    X_trains[target]   = X_tr
    y_trains[target]   = y_tr
    X_tests[target]    = np.array(test_data[target][0], dtype=np.float32)
    y_tests[target]    = np.array(test_data[target][1], dtype=np.float32)
    test_metas[target] = test_data[target][2]

    print(f"{target:<28} Train {X_trains[target].shape}  Test {X_tests[target].shape}")

temp_moy_quotidienne         Train (91747, 36, 35)  Test (1098, 36, 35)
temp_max_quotidienne         Train (91747, 36, 35)  Test (1098, 36, 35)
temp_min_quotidienne         Train (91747, 36, 35)  Test (1098, 36, 35)


In [18]:
rng = np.random.default_rng(SEED)
X_vals, y_vals = {}, {}

for target in TARGETS:
    idx = rng.permutation(len(X_trains[target]))
    X_trains[target] = X_trains[target][idx]
    y_trains[target] = y_trains[target][idx]

    val_n = int(0.15 * len(X_trains[target]))
    X_vals[target]   = X_trains[target][:val_n]
    y_vals[target]   = y_trains[target][:val_n]
    X_trains[target] = X_trains[target][val_n:]
    y_trains[target] = y_trains[target][val_n:]

    print(f"{target:<28} Train {len(X_trains[target]):>7,}  Val {len(X_vals[target]):>6,}  Test {len(X_tests[target]):>6,}")

temp_moy_quotidienne         Train  77,985  Val 13,762  Test  1,098
temp_max_quotidienne         Train  77,985  Val 13,762  Test  1,098
temp_min_quotidienne         Train  77,985  Val 13,762  Test  1,098


---
## 5. Architecture LSTM (2 couches)

In [19]:
def build_lstm(
    window: int,
    n_features: int,
    horizon: int,
    units: list = [128, 64],
    dropout: float = 0.2,
    name: str = "LSTM_model",
) -> keras.Model:
    """
    LSTM 2 couches empilées :
      Input(window, n_features)
      → LSTM(units[0], return_sequences=True) → Dropout
      → LSTM(units[1], return_sequences=False) → Dropout
      → Dense(32, relu)
      → Dense(horizon, linear)
    """
    inp = keras.Input(shape=(window, n_features), name="input")

    x = layers.LSTM(units[0], return_sequences=True, name="lstm_1")(inp)
    x = layers.Dropout(dropout, name="drop_1")(x)
    x = layers.LSTM(units[1], return_sequences=False, name="lstm_2")(x)
    x = layers.Dropout(dropout, name="drop_2")(x)
    x = layers.Dense(32, activation="relu", name="dense_hidden")(x)
    out = layers.Dense(horizon, activation="linear", name="output")(x)

    model = keras.Model(inp, out, name=name)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="huber",
        metrics=["mae"],
    )
    return model


# Aperçu de l'architecture (identique pour les 3 modèles)
demo = build_lstm(WINDOW, N_FEATURES, HORIZON, LSTM_UNITS, DROPOUT, name="demo")
demo.summary()
del demo

Model: "demo"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 36, 35)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 36, 128)        │        83,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_1 (Dropout)                │ (None, 36, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_2 (Dropout)                │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_hidden (Dense)            │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 12)             │           396 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 135,852 (530.67 KB)

 Trainable params: 135,852 (530.67 KB)

 Non-trainable params: 0 (0.00 B)

---
## 6. Entraînement des 3 modèles

In [20]:
models    = {}
histories = {}

for target in TARGETS:
    print("\n" + "=" * 60)
    print(f"  Entraînement : {target}")
    print("=" * 60)

    model_name = target.replace("_", "-")
    save_path  = MODELS_DIR / f"lstm_{model_name}.keras"

    model = build_lstm(
        window=WINDOW,
        n_features=N_FEATURES,
        horizon=HORIZON,
        units=LSTM_UNITS,
        dropout=DROPOUT,
        name=model_name,
    )

    cb_list = [
        callbacks.EarlyStopping(
            monitor="val_loss", patience=20,
            restore_best_weights=True, verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5,
            patience=10, min_lr=1e-6, verbose=1
        ),
        callbacks.ModelCheckpoint(
            filepath=str(save_path),
            monitor="val_loss", save_best_only=True, verbose=0
        ),
    ]

    # ── Pipeline tf.data avec prefetching (CPU/GPU overlap) ─────
    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = (
        tf.data.Dataset.from_tensor_slices((X_trains[target], y_trains[target]))
        .shuffle(buffer_size=min(10_000, len(X_trains[target])), seed=SEED)
        .batch(BATCH_SIZE)
        .cache()                   # mise en cache RAM après 1ère époque
        .prefetch(AUTOTUNE)        # prépare le prochain batch pendant le calcul
    )
    val_ds = (
        tf.data.Dataset.from_tensor_slices((X_vals[target], y_vals[target]))
        .batch(BATCH_SIZE)
        .cache()
        .prefetch(AUTOTUNE)
    )

    hist = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=MAX_EPOCHS,
        callbacks=cb_list,
        verbose=1,
    )

    models[target]    = model
    histories[target] = hist
    print(f"  → Modèle sauvegardé : {save_path}")
    print(f"  → Meilleure val_loss : {min(hist.history['val_loss']):.4f}")

    # Libération mémoire entre chaque modèle
    free_memory(target)


  Entraînement : temp_moy_quotidienne
Epoch 1/100
305/305 ━━━━━━━━━━━━━━━━━━━━ 55s 174ms/step - loss: 0.0245 - mae: 0.1636 - val_loss: 0.0127 - val_mae: 0.1184 - learning_rate: 0.0010
Epoch 2/100
305/305 ━━━━━━━━━━━━━━━━━━━━ 55s 182ms/step - loss: 0.0131 - mae: 0.1214 - val_loss: 0.0105 - val_mae: 0.1059 - learning_rate: 0.0010
Epoch 3/100
305/305 ━━━━━━━━━━━━━━━━━━━━ 62s 203ms/step - loss: 0.0112 - mae: 0.1114 - val_loss: 0.0091 - val_mae: 0.0977 - learning_rate: 0.0010
Epoch 4/100
305/305 ━━━━━━━━━━━━━━━━━━━━ 68s 223ms/step - loss: 0.0102 - mae: 0.1058 - val_loss: 0.0085 - val_mae: 0.0932 - learning_rate: 0.0010
Epoch 5/100
305/305 ━━━━━━━━━━━━━━━━━━━━ 69s 227ms/step - loss: 0.0096 - mae: 0.1022 - val_loss: 0.0081 - val_mae: 0.0906 - learning_rate: 0.0010
Epoch 6/100
305/305 ━━━━━━━━━━━━━━━━━━━━ 67s 219ms/step - loss: 0.0091 - mae: 0.0993 - val_loss: 0.0076 - val_mae: 0.0876 - learning_rate: 0.0010
Epoch 7/100
305/305 ━━━━━━━━━━━━━━━━━━━━ 90s 246ms/step - loss: 0.0088 - mae: 0.0971 

KeyboardInterrupt: 

---
## 7. Courbes d'apprentissage (3 modèles)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 12))

LABELS = {
    "temp_moy_quotidienne": "Temp. Moyenne",
    "temp_max_quotidienne": "Temp. Max",
    "temp_min_quotidienne": "Temp. Min",
}
COLORS = {
    "temp_moy_quotidienne": "steelblue",
    "temp_max_quotidienne": "tomato",
    "temp_min_quotidienne": "mediumseagreen",
}

for row, target in enumerate(TARGETS):
    hist = histories[target].history
    c    = COLORS[target]
    lbl  = LABELS[target]

    # Loss
    axes[row, 0].plot(hist["loss"],     color=c,      linewidth=2, label="Train")
    axes[row, 0].plot(hist["val_loss"], color=c, alpha=0.5, linewidth=2,
                      linestyle="--", label="Validation")
    axes[row, 0].set_title(f"{lbl} — Huber Loss")
    axes[row, 0].set_ylabel("Loss")
    axes[row, 0].legend()
    axes[row, 0].grid(True, alpha=0.4)

    # MAE
    axes[row, 1].plot(hist["mae"],     color=c,      linewidth=2, label="Train")
    axes[row, 1].plot(hist["val_mae"], color=c, alpha=0.5, linewidth=2,
                      linestyle="--", label="Validation")
    axes[row, 1].set_title(f"{lbl} — MAE (normalisée)")
    axes[row, 1].set_ylabel("MAE")
    axes[row, 1].legend()
    axes[row, 1].grid(True, alpha=0.4)

for ax in axes[-1]:
    ax.set_xlabel("Époque")

plt.suptitle("Courbes d'apprentissage — 3 modèles LSTM", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(MODELS_DIR / "lstm_multi_learning_curves.png", dpi=120, bbox_inches="tight")
plt.show()

---
## 8. Prédictions et dénormalisation

In [ ]:
def mape(y_true, y_pred, eps=0.1):
    mask = np.abs(y_true) > eps
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


preds_inv   = {}   # preds_inv[target]  = liste de tableaux (horizon,) par station
truths_inv  = {}   # truths_inv[target] = idem

for target in TARGETS:
    model    = models[target]
    meta     = test_metas[target]
    X_te     = X_tests[target]
    y_te     = y_tests[target]

    y_pred_sc = model.predict(X_te, verbose=0)

    pred_list, true_list = [], []
    for i, m in enumerate(meta):
        poste = m["NUM_POSTE"]
        if poste not in scalers:
            continue
        sc = scalers[poste][target]
        pred_list.append(sc.inverse_transform(y_pred_sc[i].reshape(-1, 1)).flatten())
        true_list.append(sc.inverse_transform(y_te[i].reshape(-1, 1)).flatten())

    preds_inv[target]  = pred_list
    truths_inv[target] = true_list

print("Dénormalisation terminée pour les 3 cibles.")

---
## 9. Métriques d'évaluation

In [ ]:
metrics_rows = []

for target in TARGETS:
    y_pred_all = np.concatenate(preds_inv[target])
    y_true_all = np.concatenate(truths_inv[target])

    mae_v  = mean_absolute_error(y_true_all, y_pred_all)
    rmse_v = np.sqrt(mean_squared_error(y_true_all, y_pred_all))
    r2_v   = r2_score(y_true_all, y_pred_all)
    mape_v = mape(y_true_all, y_pred_all)

    metrics_rows.append({
        "Cible"   : LABELS[target],
        "MAE (°C)": round(mae_v, 3),
        "RMSE (°C)": round(rmse_v, 3),
        "MAPE (%)" : round(mape_v, 2),
        "R²"       : round(r2_v, 4),
    })

df_metrics = pd.DataFrame(metrics_rows).set_index("Cible")

print("\n" + "=" * 58)
print("   MÉTRIQUES D'ÉVALUATION — TEST 2023-2025")
print("=" * 58)
display(df_metrics)
print("=" * 58)

In [ ]:
# Erreur MAE par pas de temps (M+1 à M+12)
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)

for ax, target in zip(axes, TARGETS):
    pred_mat  = np.array(preds_inv[target])   # (n_stations, 12)
    true_mat  = np.array(truths_inv[target])

    mae_steps  = [mean_absolute_error(true_mat[:, s], pred_mat[:, s]) for s in range(HORIZON)]
    rmse_steps = [np.sqrt(mean_squared_error(true_mat[:, s], pred_mat[:, s])) for s in range(HORIZON)]

    x = range(1, HORIZON + 1)
    ax.plot(x, mae_steps,  marker="o", label="MAE",  linewidth=2, color=COLORS[target])
    ax.plot(x, rmse_steps, marker="s", label="RMSE", linewidth=2,
            color=COLORS[target], linestyle="--", alpha=0.7)
    ax.set_title(f"{LABELS[target]}\nErreur par horizon")
    ax.set_xlabel("Mois à l'avance")
    ax.set_ylabel("Erreur (°C)")
    ax.set_xticks(list(x))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.4)

plt.suptitle("MAE & RMSE par horizon de prédiction (M+1 à M+12)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(MODELS_DIR / "lstm_multi_error_by_horizon.png", dpi=120, bbox_inches="tight")
plt.show()

---
## 10. Visualisation : prédit vs réel (6 stations)

In [ ]:
N_DISPLAY   = min(6, len(preds_inv[TARGETS[0]]))
station_idx = np.linspace(0, len(preds_inv[TARGETS[0]]) - 1, N_DISPLAY, dtype=int)
x_labels    = [f"M+{i+1}" for i in range(HORIZON)]

fig, axes = plt.subplots(N_DISPLAY, 3, figsize=(16, N_DISPLAY * 3), sharey=False)

for row, s_idx in enumerate(station_idx):
    poste = test_metas[TARGETS[0]][s_idx]["NUM_POSTE"]
    s_name = df.loc[df["NUM_POSTE"] == poste, "nom_station"].iloc[0] \
             if (df["NUM_POSTE"] == poste).any() else str(poste)

    for col, target in enumerate(TARGETS):
        ax   = axes[row, col]
        pred = preds_inv[target][s_idx]
        true = truths_inv[target][s_idx]
        c    = COLORS[target]

        ax.plot(x_labels, true, marker="o", color=c,     linewidth=2, label="Réel")
        ax.plot(x_labels, pred, marker="s", color="gray", linewidth=2,
                linestyle="--", label="Prédit")

        mae_s = mean_absolute_error(true, pred)
        title = f"{s_name}" if col == 1 else ""
        ax.set_title(f"{LABELS[target]}\nMAE = {mae_s:.2f} °C", fontsize=8)
        ax.set_ylabel("°C" if col == 0 else "")
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)
        ax.tick_params(axis="x", rotation=45, labelsize=7)

    axes[row, 1].set_title(
        f"{s_name}\n{LABELS[TARGETS[1]]} — MAE = {mean_absolute_error(truths_inv[TARGETS[1]][s_idx], preds_inv[TARGETS[1]][s_idx]):.2f} °C",
        fontsize=8
    )

plt.suptitle("Prédit vs Réel — 3 cibles × 6 stations (2023–2025)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(MODELS_DIR / "lstm_multi_pred_vs_real.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Scatter réel vs prédit pour les 3 cibles
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, target in zip(axes, TARGETS):
    y_pred_all = np.concatenate(preds_inv[target])
    y_true_all = np.concatenate(truths_inv[target])

    mae_v  = mean_absolute_error(y_true_all, y_pred_all)
    r2_v   = r2_score(y_true_all, y_pred_all)

    ax.scatter(y_true_all, y_pred_all, alpha=0.25, s=6, color=COLORS[target])
    lims = [
        min(y_true_all.min(), y_pred_all.min()) - 1,
        max(y_true_all.max(), y_pred_all.max()) + 1,
    ]
    ax.plot(lims, lims, "k--", linewidth=1.5, alpha=0.6, label="y = x")
    ax.set_title(f"{LABELS[target]}\nMAE={mae_v:.2f}°C | R²={r2_v:.3f}")
    ax.set_xlabel("Réel (°C)")
    ax.set_ylabel("Prédit (°C)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_aspect("equal", adjustable="box")

plt.suptitle("Scatter Réel vs Prédit — Tous horizons",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(MODELS_DIR / "lstm_multi_scatter.png", dpi=120, bbox_inches="tight")
plt.show()

---
## 11. Récapitulatif final

In [ ]:
print("\n" + "=" * 65)
print("          SYNTHÈSE FINALE — LSTM MULTI-SORTIES")
print("=" * 65)
print(f"  Architecture  : LSTM({LSTM_UNITS[0]}) → Dropout({DROPOUT}) → LSTM({LSTM_UNITS[1]}) → Dropout({DROPOUT}) → Dense(12)")
print(f"  Fenêtre input : {WINDOW} mois  |  Horizon : {HORIZON} mois")
print(f"  Train : 2001–{TRAIN_END}  |  Test : {TEST_START}–2025")
print(f"  Features      : {N_FEATURES}")
print(f"  Stratégie     : 3 modèles indépendants (1 par cible)")
print("=" * 65)
display(df_metrics)
print("=" * 65)
print("\nFichiers sauvegardés :")
for target in TARGETS:
    p = MODELS_DIR / f"lstm_{target.replace('_', '-')}.keras"
    print(f"  {p}")
print("\nGraphiques sauvegardés :")
for fname in ["lstm_multi_learning_curves.png",
              "lstm_multi_error_by_horizon.png",
              "lstm_multi_pred_vs_real.png",
              "lstm_multi_scatter.png"]:
    print(f"  {MODELS_DIR / fname}")